In [ ]:
# === auto-inserted: bev-solving src on path ===
import sys, pathlib
_root = pathlib.Path.cwd()
while _root != _root.parent and not (_root / 'src' / 'geometry.py').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# train_v6_dinov2_lss_bev_cleaned

Autonomous training notebook for a stronger `DINOv2-B + LSS + BEV encoder` model.
It keeps the cleaned pipeline and `test-matched` validation split from `train_v4_dinov2_cleaned`,
but replaces the old hard-projection BEV path with a depth-aware learned lift-splat path and a
stronger BEV encoder-decoder.


In [ ]:
from src.utils.dist import barrier, cleanup_distributed, get_local_rank, get_rank, get_world_size, is_dist_enabled, is_main_process, setup_distributed

%load_ext autoreload
%autoreload 2

import os, copy, time, json, math, random, gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision import transforms
from PIL import Image, ImageFile, ImageFilter
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

DATA_TRAIN = Path('/home/jupyter/project/autonomy_yandex_dataset_train')
DATA_VAL   = Path('/home/jupyter/project/autonomy_yandex_dataset_val')
DATA_TEST  = Path('/home/jupyter/project/autonomy_yandex_dataset_test')

cfg = {
    'run_dir': '/home/jupyter/project/pray/kaggle_kernel_output/runs/v6_dinov2_lss_bev_cleaned',
    'img_hw': (392, 700),
    'batch_size': 1,
    'val_batch_size': 1,
    'grad_accum': 24,
    'epochs': 30,
    'warmup_epochs': 2,
    'freeze_backbone_epochs': 2,
    'unfreeze_last_blocks': 2,
    'lr_backbone': 8e-6,
    'lr_image_neck': 8e-5,
    'lr_lss_bev': 1.2e-4,
    'weight_decay': 1e-4,
    'embed_weight_decay': 1e-2,
    'pos_weight': 5.0,
    'num_workers': 4,
    'val_num_workers': 0,
    'seed': 42,
    'ema_decay': 0.995,
    'val_target_size': 200,
    'min_rover_count': 30,
    'topk_rovers': 25,
    'rover_emb_dim': 8,
    'rover_cond_dim': 8,
    'mae_dedup_thr': 0.02,
    'dedup_camera': '/camera/inner/frontal/middle',
    'use_clean_cache': True,
    'backbone_name': 'dinov2_vitb14',
    'hub_repo': 'facebookresearch/dinov2',
    'backbone_out_dim': 768,
    'patch_size': 14,
    'tap_layers': [2, 5, 8, 11],
    'neck_dim': 128,
    'context_dim': 80,
    'depth_bins': 24,
    'depth_min': 1.0,
    'depth_max': 80.0,
    'world_z_min': -2.0,
    'world_z_max': 4.5,
    'bev_base_channels': 96,
    'bev_gn_groups': 8,
    'warm_start_ckpt': '/home/jupyter/project/pray/kaggle_kernel_output/runs/v4_dinov2_cleaned/ema_best.pt',
    'resume_training': True,
    'resume_ckpt': '/home/jupyter/project/pray/kaggle_kernel_output/runs/v6_dinov2_lss_bev_cleaned/last.pt',
    'use_ddp': True,
    'use_amp': True,
}

RUN_DIR = Path(cfg['run_dir'])
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = RUN_DIR / 'preproc_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
TORCH_HUB_DIR = RUN_DIR / 'torch_hub'
TORCH_HUB_DIR.mkdir(parents=True, exist_ok=True)
torch.hub.set_dir(str(TORCH_HUB_DIR))

setup_distributed()

random.seed(cfg['seed'] + get_rank())
np.random.seed(cfg['seed'] + get_rank())
torch.manual_seed(cfg['seed'] + get_rank())

if torch.cuda.is_available():
    device = torch.device(f'cuda:{get_local_rank()}') if is_dist_enabled() else torch.device('cuda')
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

if is_main_process():
    print('device:', device)
    if torch.cuda.is_available():
        print('cuda devices visible:', torch.cuda.device_count())
        for i in range(torch.cuda.device_count()):
            try:
                print(i, torch.cuda.get_device_name(i))
            except Exception as e:
                print(i, f'<cuda init error: {e}>')
    print('world_size:', get_world_size(), '| rank:', get_rank(), '| local_rank:', get_local_rank())
    print('torch hub dir:', TORCH_HUB_DIR)
    print('img_hw:', cfg['img_hw'])
    print('train batch_size(per gpu):', cfg['batch_size'], '| train workers:', cfg['num_workers'])
    print('val batch_size:', cfg['val_batch_size'], '| val workers:', cfg['val_num_workers'])
    print('warm_start_ckpt:', cfg['warm_start_ckpt'])
    if torch.cuda.is_available() and torch.cuda.device_count() > 1 and not is_dist_enabled():
        print('NOTE: multiple GPUs are visible, but DDP is not active because torchrun env vars were not provided.')

if is_main_process():
    with open(RUN_DIR / 'config.json', 'w') as f:
        json.dump(json.loads(json.dumps(cfg, default=str)), f, indent=2)


In [ ]:
from src.geometry import load_info_with_root, resolve_info_path, resolve_row_path

CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.cleaning import build_img_hash, clean_merged_info, compute_gt_stats, smart_deduplicate


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.splits import build_rover_vocab_from_train, encode_rover, make_test_matched_split_target


In [5]:
clean_info, dedup_report, clean_summary = clean_merged_info(
    DATA_TRAIN,
    DATA_VAL,
    cache_dir=CACHE_DIR,
    mae_thr=cfg['mae_dedup_thr'],
    dedup_camera=cfg['dedup_camera'],
    use_cache=cfg['use_clean_cache'],
)
print(json.dumps(clean_summary, indent=2, ensure_ascii=False))
display(clean_info.head(3))
if len(dedup_report):
    display(dedup_report.head(10))

train_idx, val_idx = make_test_matched_split_target(
    clean_info,
    DATA_TEST / 'info.csv',
    target_val_size=cfg['val_target_size'],
    seed=cfg['seed'],
    cache_path=CACHE_DIR / 'test_matched_val200_split.npz',
)
print('train_idx:', len(train_idx), 'val_idx:', len(val_idx))

train_info = clean_info.iloc[train_idx].reset_index(drop=True).copy()
val_info = clean_info.iloc[val_idx].reset_index(drop=True).copy()

rover_vocab, rover_stats = build_rover_vocab_from_train(
    train_info,
    min_count=cfg['min_rover_count'],
    topk=cfg['topk_rovers'],
)
rover_stats.to_csv(RUN_DIR / 'rover_embedding_stats.csv', index=False)
print('num rover classes including Other:', len(rover_vocab))
print('top rover mapping:', rover_vocab)
display(rover_stats.head(30))


Smart dedup groups: 100%|██████████| 1473/1473 [02:32<00:00,  9.66it/s]


{
  "merged_before_clean": 5000,
  "removed_empty_gt": 117,
  "after_empty_filter": 4883,
  "removed_by_dedup": 390,
  "clean_total": 4493,
  "dedup_groups": 173
}


,gt_occupancy_grid,header_ts,log_time,message_ts,/camera/inner/frontal/middle,/side/left/forward,/side/right/forward,/camera/inner/frontal/far,/camera/inner/frontal/middle/intrinsic_params,/camera/inner/frontal/middle/car_to_cam,/side/left/forward/intrinsic_params,/side/left/forward/car_to_cam,/side/right/forward/intrinsic_params,/side/right/forward/car_to_cam,/camera/inner/frontal/far/intrinsic_params,/camera/inner/frontal/far/car_to_cam,predicted_occupancy_grid,ride_date,ride_time,rover,scene_part_order,__data_root,__source_split,coverage,valid_count,pos_count
0,autonomy_yandex_dataset_train/static_grids/163...,1633857774533809000,12:18:58,1633857774533809000,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/predicted_static...,2021-10-10,12:18:58,orvy,0,/home/jupyter/project/autonomy_yandex_dataset_...,train,0.061972,1468,602
1,autonomy_yandex_dataset_train/static_grids/163...,1636812143899937000,15:34:56,1636812143899937000,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/predicted_static...,2021-11-13,15:34:56,shelly,0,/home/jupyter/project/autonomy_yandex_dataset_...,train,0.227077,5379,3752
2,autonomy_yandex_dataset_train/static_grids/163...,1633600207233930000,12:28:29,1633600207233930000,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/predicted_static...,2021-10-07,12:28:29,orvy,0,/home/jupyter/project/autonomy_yandex_dataset_...,train,0.225008,5330,1264


,kept_row_id,group_size,members
0,2271,2,"[2271, 3683]"
1,4101,2,"[4473, 4101]"
2,1366,3,"[306, 2710, 1366]"
3,98,5,"[1721, 2885, 209, 3630, 98]"
4,4151,2,"[4824, 4151]"
5,1703,3,"[526, 277, 1703]"
6,4867,3,"[4332, 4815, 4867]"
7,2549,2,"[3137, 2549]"
8,2112,2,"[2112, 3216]"
9,2330,2,"[2938, 2330]"


train_idx: 4273 val_idx: 220
num rover classes including Other: 26
top rover mapping: {'__other__': 0, 'orvy': 1, 'shelly': 2, 'lerita': 3, 'ward': 4, 'ravine': 5, 'greben': 6, 'lucky': 7, 'miro': 8, 'benzon': 9, 'natelio': 10, 'darron': 11, 'greton': 12, 'jurgie': 13, 'onipa': 14, 'targi': 15, 'kynde': 16, 'soan': 17, 'baland': 18, 'mika': 19, 'crozby': 20, 'litov': 21, 'hetera': 22, 'hankie': 23, 'stelard': 24, 'nikena': 25}


,rover,count,embedding_id,bucket
0,orvy,638,1,unique
1,shelly,496,2,unique
2,lerita,327,3,unique
3,ward,239,4,unique
4,ravine,208,5,unique
5,greben,194,6,unique
6,lucky,187,7,unique
7,miro,120,8,unique
8,benzon,114,9,unique
9,natelio,108,10,unique


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.data import BEVDatasetV4Clean


In [ ]:
from src.models.lss import ASPP2d, ConvGNAct, LSSViewTransform2D, ResidualBlock2d, _DINOv2MultiScaleBackbone, gn_groups

class StrongBEVEncoderDecoder(nn.Module):
    def __init__(self, in_c: int, base_c: int = 96, groups: int = 8):
        super().__init__()
        self.stem = nn.Sequential(
            ConvGNAct(in_c, base_c, k=3, s=1, p=1, groups=groups, act=True),
            ResidualBlock2d(base_c, base_c, stride=1, groups=groups),
        )
        self.down1 = nn.Sequential(
            ResidualBlock2d(base_c, base_c * 2, stride=2, groups=groups),
            ResidualBlock2d(base_c * 2, base_c * 2, stride=1, groups=groups),
        )
        self.down2 = nn.Sequential(
            ResidualBlock2d(base_c * 2, base_c * 4, stride=2, groups=groups),
            ResidualBlock2d(base_c * 4, base_c * 4, stride=1, groups=groups),
        )
        self.aspp = ASPP2d(base_c * 4, base_c * 4, rates=(1, 3, 6), groups=groups)
        self.up1 = nn.Sequential(
            ConvGNAct(base_c * 4 + base_c * 2, base_c * 2, k=3, s=1, p=1, groups=groups, act=True),
            ResidualBlock2d(base_c * 2, base_c * 2, stride=1, groups=groups),
        )
        self.up0 = nn.Sequential(
            ConvGNAct(base_c * 2 + base_c, base_c, k=3, s=1, p=1, groups=groups, act=True),
            ResidualBlock2d(base_c, base_c, stride=1, groups=groups),
        )
        self.head = nn.Conv2d(base_c, 1, 1)

    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.down1(s0)
        s2 = self.down2(s1)
        b = self.aspp(s2)
        u1 = self.up1(torch.cat([F.interpolate(b, size=s1.shape[-2:], mode='bilinear', align_corners=False), s1], dim=1))
        u0 = self.up0(torch.cat([F.interpolate(u1, size=s0.shape[-2:], mode='bilinear', align_corners=False), s0], dim=1))
        return self.head(u0)

class MultiCamBEVv6DINOv2LSSClean(nn.Module):
    def __init__(self, num_rover_classes: int,
                 rover_emb_dim: int = 8,
                 rover_cond_dim: int = 8,
                 n_cameras: int = 4,
                 freeze_backbone: bool = False,
                 hub_repo: str = 'facebookresearch/dinov2',
                 backbone_name: str = 'dinov2_vitb14',
                 backbone_out_dim: int = 768,
                 patch_size: int = 14,
                 tap_layers=(2, 5, 8, 11),
                 neck_dim: int = 128,
                 context_dim: int = 80,
                 depth_bins: int = 24,
                 depth_min: float = 1.0,
                 depth_max: float = 80.0,
                 world_z_min: float = -2.0,
                 world_z_max: float = 4.5,
                 bev_base_channels: int = 96,
                 bev_gn_groups: int = 8):
        super().__init__()
        self.n_cameras = n_cameras
        self.rover_cond_dim = rover_cond_dim

        self.backbone = _DINOv2MultiScaleBackbone(
            hub_repo=hub_repo,
            backbone_name=backbone_name,
            out_dim=backbone_out_dim,
            patch_size=patch_size,
            tap_layers=tap_layers,
            neck_dim=neck_dim,
            groups=bev_gn_groups,
        )
        if freeze_backbone:
            for p in self.backbone.vit.parameters():
                p.requires_grad = False

        self.view_transform = LSSViewTransform2D(
            in_c=neck_dim,
            context_c=context_dim,
            depth_bins=depth_bins,
            depth_min=depth_min,
            depth_max=depth_max,
            bev_h=BEV_H,
            bev_w=BEV_W,
            bev_res=BEV_RES,
            x_range=X_RANGE,
            y_range=Y_RANGE,
            z_min=world_z_min,
            z_max=world_z_max,
            groups=bev_gn_groups,
        )

        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        nn.init.normal_(self.rover_embed.weight, std=0.02)
        self.rover_mlp = nn.Sequential(
            nn.Linear(rover_emb_dim, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, rover_cond_dim),
            nn.ReLU(inplace=True),
        )

        self.bev_decoder = StrongBEVEncoderDecoder(
            in_c=context_dim + rover_cond_dim,
            base_c=bev_base_channels,
            groups=bev_gn_groups,
        )

    def forward_debug(self, images, intrinsics, car2cams, rover_ids):
        B, N, C, H, W = images.shape
        assert N == self.n_cameras
        x = images.reshape(B * N, C, H, W)
        back = self.backbone(x)
        fused = back['fused'].reshape(B, N, back['fused'].shape[1], back['fused'].shape[2], back['fused'].shape[3])
        bev, vt_debug = self.view_transform(fused, intrinsics, car2cams, image_hw=(H, W))
        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, BEV_H, BEV_W)
        logits = self.bev_decoder(torch.cat([bev, rover_map], dim=1))
        return {
            'tap_features': back['tap_features'],
            'image_fused': fused,
            'depth_logits': vt_debug['depth_logits'],
            'depth_prob': vt_debug['depth_prob'],
            'bev_raw': vt_debug['bev'],
            'valid_ratio': vt_debug['valid_ratio'],
            'logits': logits,
        }

    def forward(self, images, intrinsics, car2cams, rover_ids):
        dbg = self.forward_debug(images, intrinsics, car2cams, rover_ids)
        logits = torch.nan_to_num(dbg['logits'], nan=0.0, posinf=0.0, neginf=0.0)
        return logits

def load_warm_start(model: nn.Module, ckpt_path: str | Path):
    ckpt_path = Path(ckpt_path)
    if not ckpt_path.exists():
        print('warm-start checkpoint not found, starting from random init:', ckpt_path)
        return {'loaded': 0, 'skipped': 0, 'missing': None, 'unexpected': None}

    ckpt = torch.load(ckpt_path, map_location='cpu')
    state = ckpt['ema'] if 'ema' in ckpt else ckpt.get('model', ckpt)
    cur = model.state_dict()
    loadable = {}
    skipped = []
    for k, v in state.items():
        if k in cur and cur[k].shape == v.shape:
            loadable[k] = v
        else:
            skipped.append(k)
    missing, unexpected = model.load_state_dict(loadable, strict=False)
    print('loaded warm-start from', ckpt_path)
    print('  matched keys:', len(loadable))
    print('  skipped old keys:', len(skipped))
    print('  missing in new model:', len(missing))
    print('  unexpected during load:', len(unexpected))
    if len(skipped):
        print('  sample skipped:', skipped[:10])
    return {
        'loaded': len(loadable),
        'skipped': len(skipped),
        'missing': missing,
        'unexpected': unexpected,
    }

def load_resume_state(core_model, ema_model, optimizer, scheduler, scaler, run_dir: Path):
    resume_ckpt = Path(cfg.get('resume_ckpt', ''))
    if not cfg.get('resume_training', False) or not resume_ckpt.exists():
        return {
            'enabled': False,
            'start_epoch': 0,
            'best_iou': -1.0,
            'best_ema_iou': -1.0,
            'log': [],
            'elapsed_minutes': 0.0,
        }

    ckpt = torch.load(resume_ckpt, map_location='cpu')
    core_model.load_state_dict(ckpt['model'], strict=False)
    if 'ema' in ckpt:
        ema_model.load_state_dict(ckpt['ema'], strict=False)
    if 'optimizer' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer'])
    if 'scheduler' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler'])
    if 'scaler' in ckpt and ckpt['scaler'] is not None:
        scaler.load_state_dict(ckpt['scaler'])

    start_epoch = int(ckpt.get('epoch', -1)) + 1
    log_path = run_dir / 'log.csv'
    log_rows = []
    elapsed_minutes = 0.0
    if log_path.exists():
        log_rows = pd.read_csv(log_path).to_dict('records')
        if len(log_rows):
            elapsed_minutes = float(log_rows[-1].get('minutes', 0.0) or 0.0)

    best_iou = float(ckpt.get('best_iou', -1.0))
    best_ema_iou = float(ckpt.get('best_ema_iou', -1.0))
    best_path = run_dir / 'best.pt'
    ema_best_path = run_dir / 'ema_best.pt'
    if best_path.exists():
        try:
            best_iou = max(best_iou, float(torch.load(best_path, map_location='cpu').get('best_iou', -1.0)))
        except Exception:
            pass
    if ema_best_path.exists():
        try:
            best_ema_iou = max(best_ema_iou, float(torch.load(ema_best_path, map_location='cpu').get('best_ema_iou', -1.0)))
        except Exception:
            pass

    print('resumed from', resume_ckpt)
    print('  start_epoch:', start_epoch)
    print('  best_iou so far:', best_iou)
    print('  best_ema_iou so far:', best_ema_iou)
    print('  prior log rows:', len(log_rows))
    return {
        'enabled': True,
        'start_epoch': start_epoch,
        'best_iou': best_iou,
        'best_ema_iou': best_ema_iou,
        'log': log_rows,
        'elapsed_minutes': elapsed_minutes,
    }


In [ ]:
from src.losses import CompoundLossV2, _lovasz_grad, lovasz_hinge_flat
from src.metrics import iou_binary_batch
from src.utils.training import cleanup_cuda, unwrap_model

@torch.inference_mode()
def evaluate_iou(model, loader, threshold=0.5, desc='val@0.5'):
    model.eval()
    inter = 0
    union = 0
    pbar = tqdm(loader, desc=desc, leave=False)
    for batch in pbar:
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda' and cfg['use_amp'])):
            logits = model(images, intr, c2c, rover_id)
        logits = logits.float()
        i, u = iou_binary_batch(logits, gt, threshold=threshold)
        inter += i
        union += u
        pbar.set_postfix(iou=f'{inter / max(union, 1):.4f}')
    return inter / max(union, 1)


In [17]:
ds_train_warm = BEVDatasetV4Clean(train_info, mode='train', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)
ds_train_aug = BEVDatasetV4Clean(train_info, mode='train', img_hw=cfg['img_hw'], aug=True, rover_vocab=rover_vocab)
ds_val = BEVDatasetV4Clean(val_info, mode='val', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)

train_sampler_warm = None
train_sampler_aug = None
if is_dist_enabled():
    train_sampler_warm = DistributedSampler(ds_train_warm, num_replicas=get_world_size(), rank=get_rank(), shuffle=True, drop_last=True)
    train_sampler_aug = DistributedSampler(ds_train_aug, num_replicas=get_world_size(), rank=get_rank(), shuffle=True, drop_last=True)

loader_warm = DataLoader(
    ds_train_warm,
    batch_size=cfg['batch_size'],
    shuffle=(train_sampler_warm is None),
    sampler=train_sampler_warm,
    num_workers=cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=True,
)
loader_train = DataLoader(
    ds_train_aug,
    batch_size=cfg['batch_size'],
    shuffle=(train_sampler_aug is None),
    sampler=train_sampler_aug,
    num_workers=cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=True,
)
loader_val = None
if is_main_process():
    loader_val = DataLoader(
        ds_val,
        batch_size=cfg['val_batch_size'],
        shuffle=False,
        num_workers=cfg['val_num_workers'],
        pin_memory=(device.type == 'cuda'),
    )

if is_main_process():
    print('loader_warm batch_size:', loader_warm.batch_size, '| workers:', loader_warm.num_workers)
    print('loader_train batch_size:', loader_train.batch_size, '| workers:', loader_train.num_workers)
    if loader_val is not None:
        print('loader_val batch_size:', loader_val.batch_size, '| workers:', loader_val.num_workers)

    sample = ds_train_warm[0]
    for k, v in sample.items():
        if isinstance(v, torch.Tensor):
            print(k, tuple(v.shape), v.dtype)
        else:
            print(k, type(v), v)


loader_warm batch_size: 1 | workers: 4
loader_train batch_size: 1 | workers: 4
loader_val batch_size: 1 | workers: 0
images (4, 3, 392, 700) torch.float32
intrinsics (4, 3, 3) torch.float32
car2cams (4, 4, 4) torch.float32
rover_id () torch.int64
info_idx <class 'int'> 0
gt (1, 188, 126) torch.int64


In [ ]:
from src.utils.training import freeze_all_backbone, unfreeze_last_blocks, update_ema

base_model = MultiCamBEVv6DINOv2LSSClean(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
    freeze_backbone=False,
    hub_repo=cfg['hub_repo'],
    backbone_name=cfg['backbone_name'],
    backbone_out_dim=cfg['backbone_out_dim'],
    patch_size=cfg['patch_size'],
    tap_layers=cfg['tap_layers'],
    neck_dim=cfg['neck_dim'],
    context_dim=cfg['context_dim'],
    depth_bins=cfg['depth_bins'],
    depth_min=cfg['depth_min'],
    depth_max=cfg['depth_max'],
    world_z_min=cfg['world_z_min'],
    world_z_max=cfg['world_z_max'],
    bev_base_channels=cfg['bev_base_channels'],
    bev_gn_groups=cfg['bev_gn_groups'],
).to(device)

warm_info = {'loaded': 0, 'skipped': 0, 'missing': None, 'unexpected': None}
if not cfg.get('resume_training', False):
    warm_info = load_warm_start(base_model, cfg['warm_start_ckpt'])

freeze_all_backbone(base_model)

if is_dist_enabled():
    model = DDP(
        base_model,
        device_ids=[get_local_rank()],
        output_device=get_local_rank(),
        broadcast_buffers=False,
        find_unused_parameters=False,
    )
else:
    model = base_model

core_model = unwrap_model(model)
backbone_params = []
embed_params = []
image_neck_params = []
other_params = []
for name, p in core_model.named_parameters():
    if name.startswith('backbone.vit.'):
        backbone_params.append(p)
    elif name.startswith('rover_embed.'):
        embed_params.append(p)
    elif name.startswith('backbone.laterals.') or name.startswith('backbone.fuse.') or name.startswith('backbone.down') or name.startswith('backbone.neck_out.'):
        image_neck_params.append(p)
    else:
        other_params.append(p)

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': cfg['lr_backbone'], 'weight_decay': cfg['weight_decay']},
    {'params': image_neck_params, 'lr': cfg['lr_image_neck'], 'weight_decay': cfg['weight_decay']},
    {'params': other_params, 'lr': cfg['lr_lss_bev'], 'weight_decay': cfg['weight_decay']},
    {'params': embed_params, 'lr': cfg['lr_lss_bev'], 'weight_decay': cfg['embed_weight_decay']},
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'], eta_min=1e-6)
loss_fn = CompoundLossV2(pos_weight=cfg['pos_weight']).to(device)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda' and cfg['use_amp']))

ema_model = copy.deepcopy(core_model).to(device).eval()
for p in ema_model.parameters():
    p.requires_grad = False

resume_state = load_resume_state(core_model, ema_model, optimizer, scheduler, scaler, RUN_DIR)
if resume_state['start_epoch'] >= cfg['freeze_backbone_epochs']:
    unfreeze_last_blocks(model, n_last_blocks=cfg['unfreeze_last_blocks'])

if is_main_process():
    print('params M:', sum(p.numel() for p in core_model.parameters()) / 1e6)
    print('backbone frozen at start:', not any(p.requires_grad for p in core_model.backbone.vit.parameters()))
    print('warm_start loaded keys:', warm_info['loaded'])
    print('resume enabled:', resume_state['enabled'])
    print('resume start_epoch:', resume_state['start_epoch'])

    with torch.no_grad():
        batch = next(iter(loader_warm))
        images = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rover_id = batch['rover_id'].to(device)
        dbg = core_model.forward_debug(images, intr, c2c, rover_id)

        print('image_fused shape:', tuple(dbg['image_fused'].shape), 'finite:', torch.isfinite(dbg['image_fused']).all().item())
        print('depth_logits shape:', tuple(dbg['depth_logits'].shape), 'finite:', torch.isfinite(dbg['depth_logits']).all().item())
        print('depth_prob shape:', tuple(dbg['depth_prob'].shape), 'finite:', torch.isfinite(dbg['depth_prob']).all().item())
        print('bev_raw shape:', tuple(dbg['bev_raw'].shape), 'finite:', torch.isfinite(dbg['bev_raw']).all().item())
        print('valid_ratio:', dbg['valid_ratio'])
        print('logits shape:', tuple(dbg['logits'].shape), 'finite:', torch.isfinite(dbg['logits']).all().item())

cleanup_cuda()
barrier()


In [ ]:
log = list(resume_state['log'])
best_iou = float(resume_state['best_iou'])
best_ema_iou = float(resume_state['best_ema_iou'])
start = time.time() - float(resume_state['elapsed_minutes']) * 60.0
start_epoch = int(resume_state['start_epoch'])
backbone_unfrozen = bool(start_epoch >= cfg['freeze_backbone_epochs'])

for epoch in range(start_epoch, cfg['epochs']):
    if train_sampler_warm is not None:
        train_sampler_warm.set_epoch(epoch)
    if train_sampler_aug is not None:
        train_sampler_aug.set_epoch(epoch)

    if (not backbone_unfrozen) and epoch >= cfg['freeze_backbone_epochs']:
        unfreeze_last_blocks(model, n_last_blocks=cfg['unfreeze_last_blocks'])
        backbone_unfrozen = True
        if is_main_process():
            print(f'unfroze last {cfg["unfreeze_last_blocks"]} DINO blocks at epoch {epoch:02d}')

    model.train()
    loader = loader_warm if epoch < cfg['warmup_epochs'] else loader_train
    phase = 'WARM' if epoch < cfg['warmup_epochs'] else 'AUG'

    losses = []
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc=f'ep{epoch:02d} [{phase}]', disable=not is_main_process())
    for step, batch in enumerate(pbar):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda' and cfg['use_amp'])):
            logits = model(images, intr, c2c, rover_id)
        logits = logits.float()
        loss, parts = loss_fn(logits, gt)
        loss = loss / cfg['grad_accum']

        if not torch.isfinite(loss):
            raise RuntimeError(f'Non-finite loss detected at epoch={epoch} step={step}: {loss.item()}')

        scaler.scale(loss).backward()
        if (step + 1) % cfg['grad_accum'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(core_model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            update_ema(ema_model, model, cfg['ema_decay'])

        losses.append(loss.item() * cfg['grad_accum'])
        if is_main_process() and step % 20 == 0:
            pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.3f}', bce=f"{parts['bce']:.2f}")

    if len(loader) % cfg['grad_accum'] != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(core_model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        update_ema(ema_model, model, cfg['ema_decay'])

    scheduler.step()
    barrier()

    val_iou_05 = None
    val_iou_05_ema = None
    if is_main_process():
        cleanup_cuda()
        print('start val model @0.5')
        val_iou_05 = evaluate_iou(core_model, loader_val, threshold=0.5, desc=f'val model ep{epoch:02d}')

        cleanup_cuda()
        print('start val ema @0.5')
        val_iou_05_ema = evaluate_iou(ema_model, loader_val, threshold=0.5, desc=f'val ema ep{epoch:02d}')
        cleanup_cuda()

        elapsed = (time.time() - start) / 60
        backbone_grad_enabled = any(p.requires_grad for p in core_model.backbone.vit.parameters())

        row = {
            'epoch': epoch,
            'loss': float(np.mean(losses)) if len(losses) else None,
            'val_iou_05': float(val_iou_05),
            'val_iou_05_ema': float(val_iou_05_ema),
            'backbone_trainable': bool(backbone_grad_enabled),
            'minutes': float(elapsed),
        }

        print(
            f"ep{epoch:02d} | loss {np.mean(losses):.3f} | "
            f"val@0.5 {val_iou_05:.4f} | ema@0.5 {val_iou_05_ema:.4f} | "
            f"backbone_trainable={backbone_grad_enabled} | {elapsed:.1f}m"
        )

        if val_iou_05 > best_iou:
            best_iou = val_iou_05
            torch.save({
                'model_type': 'v6_dinov2_lss_bev_cleaned',
                'model': core_model.state_dict(),
                'epoch': epoch,
                'best_iou': float(val_iou_05),
                'best_t': 0.5,
                'rover_vocab': rover_vocab,
                'cfg': cfg,
            }, RUN_DIR / 'best.pt')
            print('  new best model:', val_iou_05)

        if val_iou_05_ema > best_ema_iou:
            best_ema_iou = val_iou_05_ema
            torch.save({
                'model_type': 'v6_dinov2_lss_bev_cleaned',
                'model': core_model.state_dict(),
                'ema': ema_model.state_dict(),
                'epoch': epoch,
                'best_ema_iou': float(val_iou_05_ema),
                'best_t_ema': 0.5,
                'rover_vocab': rover_vocab,
                'cfg': cfg,
            }, RUN_DIR / 'ema_best.pt')
            print('  new best ema:', val_iou_05_ema)

        log.append(row)
        pd.DataFrame(log).to_csv(RUN_DIR / 'log.csv', index=False)
        torch.save({
            'model_type': 'v6_dinov2_lss_bev_cleaned',
            'model': core_model.state_dict(),
            'ema': ema_model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'scaler': scaler.state_dict() if scaler is not None else None,
            'epoch': epoch,
            'best_iou': float(best_iou),
            'best_ema_iou': float(best_ema_iou),
            'rover_vocab': rover_vocab,
            'cfg': cfg,
        }, RUN_DIR / 'last.pt')

    barrier()


ep03 [AUG]: 100%|██████████| 4273/4273 [13:57<00:00,  5.10it/s, bce=0.77, loss=0.639]]


start val model @0.5


start val ema @0.5


ep03 | loss 0.621 | val@0.5 0.5780 | ema@0.5 0.5716 | backbone_trainable=True | 61.7m
  new best model: 0.5779944906509358
  new best ema: 0.5716100603978407


ep04 [AUG]: 100%|██████████| 4273/4273 [13:54<00:00,  5.12it/s, bce=0.69, loss=0.556]


...

ep09 | loss 0.525 | val@0.5 0.5935 | ema@0.5 0.5948 | backbone_trainable=True | 163.2m
  new best model: 0.5934824715060691
  new best ema: 0.5947570951195947


ep10 [AUG]: 100%|██████████| 4273/4273 [13:57<00:00,  5.10it/s, bce=0.83, loss=0.471]


start val model @0.5


start val ema @0.5


val ema ep10:  19%|█▉        | 42/220 [00:15<01:04,  2.77it/s, iou=0.5929]

### Notes

- This notebook intentionally does **not** do threshold sweep inside training.
- Use a separate inference notebook after training to compare `best.pt` and `ema_best.pt`, pick the best global threshold, and build submissions.
